In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_ViT
from train import WarmUpCosine, CustomWeightDecaySGD, RSquared
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_reg import load_wb_if_exists
from network import HierarchicalConvEmbedding, TransformerBlock, AddPositionEmbedding
from evaluate_reg import evalu_stream_main_selected, evalu_select, eval_acc_select_list_single_thresholds, evalu_prepare, compute_stats

In [ ]:
SAVE_PATH = "utkface_12k_split.npz" 
def load_dataset_if_exists(path=SAVE_PATH):
    """
    Load if file exists, otherwise return None.
    """
    if os.path.exists(path):
        data = np.load(path)
        print("✔ Cached data detected, loading directly")
        return (data["x_train"], data["y_train"],
                data["x_test"], data["y_test"])
    return None

In [5]:
x_train, y_train, x_test, y_test = load_dataset_if_exists()

✔ 检测到缓存数据，已直接加载


In [6]:
model=load_ViT()

In [7]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("dense_16").output,
    outputs=model.output
)

In [8]:
l_label = [2,3,4,5,6,7,8,9,10,13]

In [9]:
layer_list = [model.layers[l].name for l in l_label]

In [10]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data/ViT-8/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    #x_attack = np.load(o_path, allow_pickle=True)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data/ViT-8/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data/ViT-8/layer_cache/salt/"+str(i))
    save_layer_outputs_and_labels(model, x_move, y_train, layer_list, save_dir="D:/Data/ViT-8/layer_cache/move/"+str(i))
    save_layer_outputs_and_labels(model, x_occ, y_train, layer_list, save_dir="D:/Data/ViT-8/layer_cache/occ/"+str(i))

[SAVE] Generating layer outputs for: D:/Data/ViT-8/layer_cache/base
[Saved] add_position_embedding: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_1: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_2: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_3: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_4: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_5: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_6: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_7: outputs (20000, 8192), labels (20000,)
[Saved] dense_16: outputs (20000, 128), labels (20000,)
[SAVE] Generating layer outputs for: D:/Data/ViT-8/layer_cache/gauss/0
[Saved] add_position_embedding: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_1: outputs (20000, 8192), lab

In [11]:
for i in range(5):
    path = "./attack/" + str(i)
    ATTACK_DIR = os.path.join(path, "ViT_pgd.npy")
    x_attack = np.load(ATTACK_DIR)
    save_layer_outputs_and_labels(model, x_attack, y_train, layer_list, save_dir="D:/Data/ViT-8/layer_cache/attack/"+str(i))

[SAVE] Generating layer outputs for: D:/Data/ViT-8/layer_cache/attack/0
[Saved] add_position_embedding: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_1: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_2: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_3: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_4: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_5: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_6: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_7: outputs (20000, 8192), labels (20000,)
[Saved] dense_16: outputs (20000, 128), labels (20000,)
[SAVE] Generating layer outputs for: D:/Data/ViT-8/layer_cache/attack/1
[Saved] add_position_embedding: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block: outputs (20000, 8192), labels (20000,)
[Saved] transformer_block_1: outputs (20000, 8192)

In [12]:
CACHE_DIR = "./ViT-8/w_and_b_cache"

In [13]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data/ViT-8/layer_cache/base")

In [14]:
threshold_list, Y_list = evalu_prepare(y_train, n=9)

In [15]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir="D:/Data/ViT-8/layer_cache/base")

In [16]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir="D:/Data/ViT-8/layer_cache/base")
base_final = eval_acc_select_list_single_thresholds(model, x_train, 'train', select_list, threshold_list) 
base = np.concatenate((base,base_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.86113076 0.75787155 0.67486749 0.65880737 0.67052598 0.56980782
 0.59425108 0.67330541 0.82901962]
Layer 1
accuracy: [0.88595536 0.81003935 0.74662937 0.6682805  0.70776015 0.59668993
 0.60570937 0.7106485  0.82053758]
Layer 2
accuracy: [0.91542337 0.82214925 0.75962912 0.6970211  0.74970128 0.61611094
 0.58938998 0.68298958 0.81323856]
Layer 3
accuracy: [0.94269698 0.8909615  0.81696933 0.72924145 0.78483788 0.62731196
 0.57452117 0.67186838 0.81077523]
Layer 4
accuracy: [0.94601911 0.90994609 0.82130735 0.78058488 0.84226272 0.68175433
 0.58753048 0.68454076 0.81452053]
Layer 5
accuracy: [0.9472635  0.94442574 0.87106841 0.81378969 0.84742872 0.68442411
 0.66152524 0.75421532 0.82366477]
Layer 6
accuracy: [0.95448473 0.9533881  0.88855022 0.80904041 0.84148897 0.68531046
 0.6779387  0.76355992 0.82469537]
Layer 7
accuracy: [0.95013991 0.94021964 0.891105   0.82575724 0.87067793 0.7307975
 0.67377584 0.79730225 0.84717161]
Layer 8
accuracy: [0.94889745 0.94105193 

In [17]:
compute_stats(base)

(array([[0.76462327, 0.63304706, 0.6988587 ],
        [0.81420802, 0.65757686, 0.71229848],
        [0.83240058, 0.68761111, 0.69520604],
        [0.8835426 , 0.7137971 , 0.6857216 ],
        [0.89242419, 0.76820065, 0.69553059],
        [0.92091922, 0.78188084, 0.74646844],
        [0.93214102, 0.77861328, 0.755398  ],
        [0.92715485, 0.80907756, 0.7727499 ],
        [0.92188774, 0.82281011, 0.81381598],
        [0.943249  , 0.81652713, 0.8739396 ],
        [0.99986059, 0.99977307, 1.        ]]),
 array([[7.61913870e-02, 4.49720814e-02, 9.75322030e-02],
        [5.69559275e-02, 4.59715469e-02, 8.77110089e-02],
        [6.40144738e-02, 5.49424259e-02, 9.17931644e-02],
        [5.15954820e-02, 6.52303602e-02, 9.69464768e-02],
        [5.23992275e-02, 6.61098219e-02, 9.29935554e-02],
        [3.52688773e-02, 7.02673788e-02, 6.64194648e-02],
        [3.08265998e-02, 6.72918498e-02, 6.01904916e-02],
        [2.58108076e-02, 5.83111878e-02, 7.28863807e-02],
        [3.28066013e-02, 5.3

In [18]:
attack = np.zeros((len(layer_list),9))
attack_final = np.zeros(9)
for i in range(5):
    attack += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/ViT-8/layer_cache/attack/"+str(i))
attack = attack/5
for i in range(5):
    path = "./attack/" + str(i)
    ATTACK_DIR = os.path.join(path, "ViT_pgd.npy")
    attack_final += eval_acc_select_list_single_thresholds(model, ATTACK_DIR, 'train', select_list, threshold_list)
attack_final = attack_final/5
attack = np.concatenate((attack,attack_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.88476054 0.77163143 0.6887569  0.64729434 0.63347482 0.5738206
 0.62694839 0.72254875 0.85894618]
Layer 1
accuracy: [0.92386606 0.81467518 0.73980511 0.63956559 0.5479492  0.62837021
 0.71392559 0.82381978 0.91945884]
Layer 2
accuracy: [0.92387049 0.81164157 0.73469537 0.61978766 0.44019236 0.62473841
 0.72009055 0.82634753 0.9209183 ]
Layer 3
accuracy: [0.92011103 0.76738717 0.68012292 0.54670264 0.43931439 0.62234467
 0.72099568 0.82699483 0.9209183 ]
Layer 4
accuracy: [0.9064412  0.71903231 0.62752324 0.53337537 0.44710508 0.62253586
 0.72159942 0.82699483 0.9209183 ]
Layer 5
accuracy: [0.91328012 0.69878559 0.60861141 0.53130357 0.43931439 0.62137329
 0.72159942 0.82699483 0.92008789]
Layer 6
accuracy: [0.90768566 0.67693364 0.60391747 0.53105332 0.43931439 0.61974861
 0.72159942 0.82699483 0.92070979]
Layer 7
accuracy: [0.91102257 0.70653324 0.6090307  0.53155482 0.43931439 0.58963564
 0.72180915 0.82699483 0.92050129]
Layer 8
accuracy: [0.91599166 0.72158058 

In [19]:
compute_stats(attack)

(array([[0.7817376 , 0.61834152, 0.73581755],
        [0.82605989, 0.60514209, 0.81914619],
        [0.82344245, 0.56114126, 0.8225159 ],
        [0.7894418 , 0.53580595, 0.82295493],
        [0.75143385, 0.53445851, 0.82317085],
        [0.74035609, 0.53036535, 0.82278343],
        [0.72954319, 0.52966655, 0.82308745],
        [0.74245732, 0.52018098, 0.82314469],
        [0.75066608, 0.47260069, 0.82315887],
        [0.67515092, 0.42314103, 0.17812265],
        [0.67515244, 0.42418696, 0.17813318]]),
 array([[0.08041313, 0.03205329, 0.0951049 ],
        [0.07564022, 0.04009986, 0.08384436],
        [0.07767085, 0.08554029, 0.08204696],
        [0.09911032, 0.07475679, 0.081686  ],
        [0.1159398 , 0.07185343, 0.08141651],
        [0.12743939, 0.07394256, 0.08095189],
        [0.12940516, 0.07321265, 0.0813164 ],
        [0.12570957, 0.06190857, 0.0811769 ],
        [0.12514191, 0.04260825, 0.08119392],
        [0.06811508, 0.09540072, 0.03991667],
        [0.06788945, 0.09644206,

In [20]:
gauss = np.zeros((len(layer_list),9))
gauss_final = np.zeros(9)
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/ViT-8/layer_cache/gauss/"+str(i))
gauss = gauss/10
for i in range(10):
    path = "./noise/" + str(i)
    GAUSS_DIR = os.path.join(path, "gauss.npy")
    gauss_final += eval_acc_select_list_single_thresholds(model, GAUSS_DIR, 'train', select_list, threshold_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.86275325 0.72845299 0.6149546  0.58296423 0.61604862 0.56374645
 0.52247898 0.5450874  0.67619632]
Layer 1
accuracy: [0.92387049 0.81404317 0.73811704 0.63886489 0.59620197 0.62275422
 0.72094035 0.82657127 0.92070839]
Layer 2
accuracy: [0.92387049 0.81404317 0.73832554 0.63907167 0.54267279 0.62317301
 0.72159942 0.82699483 0.9209183 ]
Layer 3
accuracy: [0.92387049 0.81404317 0.73832554 0.63907167 0.5405618  0.62317301
 0.72159942 0.82699483 0.9209183 ]
Layer 4
accuracy: [0.92387049 0.81404317 0.73832554 0.63907167 0.54411823 0.62317301
 0.72159942 0.82699483 0.9209183 ]
Layer 5
accuracy: [0.92387049 0.81404317 0.73832554 0.63907167 0.54581734 0.6236134
 0.72159942 0.82699483 0.9209183 ]
Layer 6
accuracy: [0.92387049 0.81404317 0.73832554 0.63907167 0.54525755 0.62355597
 0.72159942 0.82699483 0.9209183 ]
Layer 7
accuracy: [0.92387049 0.81404317 0.73832554 0.63907167 0.54165812 0.60592504
 0.72137069 0.82699483 0.9209183 ]
Layer 8
accuracy: [0.92387049 0.81404317 

In [21]:
compute_stats(gauss)

(array([[0.73611666, 0.58589568, 0.57803865],
        [0.82541334, 0.61851092, 0.82285445],
        [0.82541307, 0.60155405, 0.82317085],
        [0.82541307, 0.60125516, 0.82317085],
        [0.82541307, 0.60217847, 0.82317085],
        [0.82541307, 0.60424981, 0.82317085],
        [0.82541307, 0.60285786, 0.82317842],
        [0.82542005, 0.59470703, 0.82300176],
        [0.82542005, 0.57306268, 0.82285589],
        [0.82552506, 0.51939231, 0.18985196],
        [0.82543434, 0.52116858, 0.20906867]]),
 array([[0.10023417, 0.02139966, 0.06908406],
        [0.07617356, 0.01890091, 0.0815026 ],
        [0.07617387, 0.04233097, 0.08141651],
        [0.07617387, 0.04273435, 0.08141651],
        [0.07617387, 0.04143116, 0.08141651],
        [0.07617387, 0.03848791, 0.08141651],
        [0.07617387, 0.03965645, 0.08140706],
        [0.07616589, 0.03909964, 0.08162754],
        [0.07616589, 0.04673307, 0.0818102 ],
        [0.07608681, 0.1079363 , 0.06937542],
        [0.07614421, 0.10806448,

In [22]:
salt = np.zeros((len(layer_list),9))
salt_final = np.zeros(9)
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/ViT-8/layer_cache/salt/"+str(i))
salt = salt/10
for i in range(10):
    path = "./noise/" + str(i)
    SALT_DIR = os.path.join(path, "salt.npy")
    salt_final += eval_acc_select_list_single_thresholds(model, SALT_DIR, 'train', select_list, threshold_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.7946701  0.64210275 0.52354498 0.53525127 0.62328612 0.44654286
 0.36481786 0.32011599 0.36829185]
Layer 1
accuracy: [0.92387049 0.8142419  0.73916237 0.64023599 0.60956508 0.62360983
 0.72073228 0.82634924 0.92070839]
Layer 2
accuracy: [0.92387049 0.81404317 0.73832554 0.63907167 0.55059712 0.62317301
 0.72159942 0.82699483 0.9209183 ]
Layer 3
accuracy: [0.92387049 0.81404317 0.73855057 0.63907167 0.55701787 0.62296725
 0.72159942 0.82699483 0.9209183 ]
Layer 4
accuracy: [0.92387049 0.81404317 0.73855057 0.63907167 0.57766695 0.62317301
 0.72159942 0.82699483 0.9209183 ]
Layer 5
accuracy: [0.92387049 0.81404317 0.73855057 0.639025   0.59459638 0.62470194
 0.72159942 0.82699483 0.9209183 ]
Layer 6
accuracy: [0.92387049 0.81404317 0.73855057 0.64043518 0.58676859 0.62683389
 0.72159942 0.82699483 0.9209183 ]
Layer 7
accuracy: [0.92387049 0.81404317 0.73876012 0.63952331 0.58570562 0.63550534
 0.72159776 0.82699483 0.9209183 ]
Layer 8
accuracy: [0.92387049 0.81404317

In [23]:
compute_stats(salt)

(array([[0.6544401 , 0.53373791, 0.34789267],
        [0.82553373, 0.62423832, 0.82266192],
        [0.82541307, 0.604062  , 0.82316323],
        [0.82542057, 0.60639724, 0.82317085],
        [0.82542057, 0.61459796, 0.82317085],
        [0.82549675, 0.62007957, 0.82317085],
        [0.8255038 , 0.61871054, 0.82317027],
        [0.82551896, 0.61958536, 0.82320925],
        [0.82563325, 0.62010355, 0.82325117],
        [0.82875126, 0.54899337, 0.39108084],
        [0.83247776, 0.56403193, 0.44050623]]),
 array([[0.11301283, 0.07382173, 0.02225593],
        [0.07605613, 0.01208607, 0.08163455],
        [0.07617387, 0.0388732 , 0.08142602],
        [0.0761653 , 0.03560171, 0.08141651],
        [0.0761653 , 0.02429572, 0.08141651],
        [0.07611473, 0.0176221 , 0.08141651],
        [0.07609687, 0.0210313 , 0.08141723],
        [0.07605943, 0.02321211, 0.0813686 ],
        [0.0759427 , 0.02727786, 0.08131632],
        [0.07384433, 0.08415739, 0.05505895],
        [0.07266646, 0.08515378,

In [24]:
move = np.zeros((len(layer_list),9))
move_final = np.zeros(9)
for i in range(10):
    move += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/ViT-8/layer_cache/move/"+str(i))
move = move/10
for i in range(10):
    path = "./noise/" + str(i)
    MOVE_DIR = os.path.join(path, "move.npy")
    move_final += eval_acc_select_list_single_thresholds(model, MOVE_DIR, 'train', select_list, threshold_list)
move_final = move_final/10
move = np.concatenate((move,move_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.92366399 0.81304279 0.73745988 0.63765829 0.56474712 0.62253401
 0.72032739 0.82572979 0.92049849]
Layer 1
accuracy: [0.91178781 0.81038639 0.73860342 0.63722278 0.62075643 0.63070849
 0.67629019 0.79436018 0.89619268]
Layer 2
accuracy: [0.90286157 0.7827245  0.69898001 0.63652066 0.62968821 0.63822313
 0.67494765 0.79220448 0.90657945]
Layer 3
accuracy: [0.92257168 0.78620954 0.69897547 0.6217513  0.60760399 0.62119528
 0.64980212 0.77310449 0.90595615]
Layer 4
accuracy: [0.92090703 0.77817828 0.68003998 0.63057278 0.61845367 0.64064507
 0.65698806 0.77783247 0.89992482]
Layer 5
accuracy: [0.92530452 0.7932391  0.68640049 0.63666322 0.61889342 0.63401064
 0.68469445 0.79700883 0.89327124]
Layer 6
accuracy: [0.92530444 0.79843447 0.68620745 0.618443   0.6090727  0.62094708
 0.67807738 0.78673903 0.88391467]
Layer 7
accuracy: [0.92551295 0.81072077 0.69694057 0.63734372 0.61095328 0.61436762
 0.64506576 0.77730586 0.87160842]
Layer 8
accuracy: [0.92550977 0.80783321

In [25]:
compute_stats(move)

(array([[0.82493797, 0.60819816, 0.82220571],
        [0.82101911, 0.63244978, 0.78987835],
        [0.79487445, 0.63683944, 0.79128859],
        [0.80301938, 0.61973727, 0.77628101],
        [0.79322836, 0.63045586, 0.77780532],
        [0.80049765, 0.63052798, 0.79074785],
        [0.80323592, 0.61952807, 0.78226067],
        [0.81115599, 0.62093342, 0.76481321],
        [0.80641376, 0.62277468, 0.77208766],
        [0.81502833, 0.60027145, 0.644666  ],
        [0.7999478 , 0.61091745, 0.72684738]]),
 array([[0.0763604 , 0.03214343, 0.08175628],
        [0.0712786 , 0.0083683 , 0.08983209],
        [0.08300767, 0.00483843, 0.0963554 ],
        [0.09226734, 0.00769173, 0.10607947],
        [0.09950129, 0.00634486, 0.09892217],
        [0.09814623, 0.00585942, 0.08490366],
        [0.09708263, 0.00545929, 0.08425295],
        [0.09332374, 0.01534269, 0.0924288 ],
        [0.09753221, 0.01868898, 0.07786908],
        [0.09184879, 0.02669524, 0.0855156 ],
        [0.10854048, 0.01537362,

In [26]:
occ = np.zeros((len(layer_list),9))
occ_final = np.zeros(9)
for i in range(10):
    occ += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data/ViT-8/layer_cache/occ/"+str(i))
occ = occ/10
for i in range(10):
    path = "./noise/" + str(i)
    OCC_DIR = os.path.join(path, "occ.npy")
    occ_final += eval_acc_select_list_single_thresholds(model, OCC_DIR, 'train', select_list, threshold_list)
occ_final = occ_final/10
occ = np.concatenate((occ,occ_final.reshape(1,9)),axis=0)

Layer 0
accuracy: [0.87313677 0.74752987 0.63372907 0.58606345 0.56471012 0.60358249
 0.65317223 0.73229953 0.86163587]
Layer 1
accuracy: [0.92426406 0.81631095 0.74623684 0.65528892 0.61776309 0.62738471
 0.71944881 0.82571776 0.92008458]
Layer 2
accuracy: [0.924279   0.81649436 0.74178495 0.64849028 0.64882295 0.62418925
 0.72092135 0.8267728  0.9209183 ]
Layer 3
accuracy: [0.924281   0.82289011 0.74834838 0.66830771 0.64231818 0.62753059
 0.72244756 0.82655986 0.9209183 ]
Layer 4
accuracy: [0.92490963 0.8276109  0.75690153 0.67277342 0.62458871 0.6304098
 0.7220281  0.82676139 0.9209183 ]
Layer 5
accuracy: [0.92574793 0.83915734 0.77623473 0.68235891 0.62633567 0.6386033
 0.72401656 0.82720544 0.92070979]
Layer 6
accuracy: [0.92553625 0.84262579 0.78155664 0.68542522 0.62206892 0.64252218
 0.72380739 0.82740698 0.92070979]
Layer 7
accuracy: [0.92512449 0.83255612 0.77579893 0.67970886 0.64154136 0.65677687
 0.72858854 0.82782085 0.92070979]
Layer 8
accuracy: [0.92532974 0.83478625 0

In [27]:
compute_stats(occ)

(array([[0.74896515, 0.5836891 , 0.74949787],
        [0.8289383 , 0.63584509, 0.82168299],
        [0.82760543, 0.63950031, 0.82315139],
        [0.83182651, 0.64560209, 0.82326743],
        [0.83618312, 0.64457062, 0.82357263],
        [0.84726499, 0.65031422, 0.8240194 ],
        [0.84988499, 0.65252501, 0.82445221],
        [0.84534261, 0.65917704, 0.82636443],
        [0.84836905, 0.66080975, 0.82515271],
        [0.85857058, 0.64862258, 0.79772808],
        [0.87798142, 0.67109846, 0.84356756]]),
 array([[0.09524684, 0.01766344, 0.08704118],
        [0.0726025 , 0.01450692, 0.08220735],
        [0.07493823, 0.01003461, 0.08136737],
        [0.07233988, 0.01581281, 0.0811879 ],
        [0.06931997, 0.02112855, 0.08089325],
        [0.06147504, 0.02329895, 0.08048218],
        [0.05950232, 0.03007708, 0.08001637],
        [0.06134553, 0.01731048, 0.07787303],
        [0.05850374, 0.01451101, 0.07937478],
        [0.06178714, 0.01602799, 0.09145504],
        [0.06201942, 0.01548177,

In [28]:
SAVE_FILE='ViT-8.pkl'

In [29]:
progress = {"base": base, "attack": attack,"gauss": gauss,"salt": salt,"move": move,"occ": occ}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [ ]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=6): 
    """ 
    Input: 
        matrix_dict: {name: np.ndarray(m, n)}, a dictionary containing 6 matrices 
    Output: 
        stats: { 
            name: { 
                "mean": (m,), 
                "min":  (m,), 
                "max":  (m,) 
            }, ... 
        } 
    For each component i, statistics are computed along axis=1 (over n samples): 
        mean[i] = mean(X[i, :]) 
        min[i]  = min(X[i, :]) 
        max[i]  = max(X[i, :]) 
    """ 
    if check_num is not None and len(matrix_dict) != check_num: 
        raise ValueError(f"Expected {check_num} matrices, but received {len(matrix_dict)}") 
 
    # shape consistency 
    shapes = [np.asarray(v).shape for v in matrix_dict.values()] 
    if len(set(shapes)) != 1: 
        raise ValueError(f"Inconsistent matrix shapes: {shapes}") 
 
    stats = {} 
    for name, X in matrix_dict.items(): 
        X = np.asarray(X) 
        stats[name] = { 
            "mean": X.mean(axis=1), 
            "std":  X.std(axis=1), 
            "min":  X.min(axis=1), 
            "max":  X.max(axis=1), 
        } 
    return stats 

In [32]:
SAVE_FILE='ViT-8.pkl'
with open(SAVE_FILE, "rb") as f:
    progress = pickle.load(f)

In [33]:
mean_var = stats_minmax_from_matrix_dict(progress)

In [34]:
mean_var

{'base': {'mean': array([0.69884301, 0.72802779, 0.73840591, 0.76102043, 0.78538514,
         0.81642283, 0.82205076, 0.83632744, 0.85283794, 0.87790524,
         0.99987789]),
  'std': array([0.0930883 , 0.09253533, 0.09802015, 0.1143494 , 0.10891968,
         0.09591525, 0.09583082, 0.08643513, 0.06393284, 0.07567531,
         0.00015262]),
  'min': array([0.56980782, 0.59668993, 0.58938998, 0.57452117, 0.58753048,
         0.66152524, 0.6779387 , 0.67377584, 0.76117416, 0.70913118,
         0.99954863]),
  'max': array([0.86113076, 0.88595536, 0.91542337, 0.94269698, 0.94601911,
         0.9472635 , 0.95448473, 0.95013991, 0.94889745, 0.97022985,
         1.        ])},
 'attack': {'mean': array([0.71196555, 0.75011606, 0.73569987, 0.71606756, 0.70302107,
         0.69783496, 0.69409906, 0.695261  , 0.68214188, 0.42547153,
         0.42582419]),
  'std': array([0.1012277 , 0.12370583, 0.14808567, 0.15425841, 0.15321739,
         0.15676707, 0.15668491, 0.15859803, 0.17564859, 0.2151